In [188]:
import sys
from pathlib import Path

# add the parent directory as a package path 
parent_dir = Path(__file__).resolve().parents[1] if '__file__' in locals() else Path('.').resolve().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

from src.graph_utils import create_layered_graph, assign_properties
from src.lp import standard_max_flow

layer_sizes = [6, 5, 3, 1, 1, 2, 5, 3, 1]
G, layers = create_layered_graph(layer_sizes)
G = assign_properties(G, layer_sizes)
result, flow_dict = standard_max_flow(G)
result


{'Problem': [{'Lower bound': 4.0, 'Upper bound': 4.0, 'Number of objectives': 1, 'Number of constraints': 0, 'Number of variables': 0, 'Sense': 'maximize'}], 'Solver': [{'Status': 'ok', 'Termination condition': 'optimal', 'Termination message': 'TerminationCondition.optimal'}]}

In [200]:
class Objectives:

    @staticmethod
    def mcf_obj(m):
        return sum(m.cost[u, v] * m.x[u, v] for (u, v) in m.edges)

    @staticmethod
    def mf_obj(m):
        return m.F


class Rules:
    @staticmethod
    def edge_cap_rule(m, u, v):
        return m.x[u, v] <= m.capacity[u, v]
    
    @staticmethod
    def edge_flow_rule(m, n):
        if n in (m.s, m.t):
            return pyo.Constraint.Skip
        inflow = sum(m.x[u, v] for (u, v) in m.edges if v == n)
        outflow = sum(m.x[u, v] for (u, v) in m.edges if u == n)
        return outflow - inflow == m.demand[n]
    
    @staticmethod
    def source_flow_rule(m):
        outflow = sum(m.x[u, v] for (u, v) in m.edges if u in m.s)
        inflow = sum(m.x[u, v] for (u, v) in m.edges if v in m.s)
        return outflow - inflow == m.F
    
    @staticmethod
    def sink_flow_rule(m):
        inflow = sum(m.x[u, v] for (u, v) in m.edges if v in m.t)
        outflow = sum(m.x[u, v] for (u, v) in m.edges if u in m.t)
        return inflow - outflow == m.F


import pyomo.environ as pyo

class LP():

    def __init__(self, G):
        self.g = G.copy()
        self.r = Rules()
        self.o = Objectives()
        self.model = pyo.ConcreteModel()
        self.g = self.__configure_graph()
        self._create_base_model()
        self.create_model(problem='max_flow')


    def _create_base_model(self):
        model = self.model
        model.nodes = pyo.Set(initialize=list(self.g.nodes()))
        model.edges = pyo.Set(initialize=list(self.g.edges()), dimen=2)
        model.s = 's'
        model.t = 't'
        model.x = pyo.Var(self.model.edges, within=pyo.NonNegativeReals)
        model.capacity = pyo.Param(model.edges,initialize={(u, v): self.g.edges[u, v]['capacity'] for u, v in self.g.edges()})
        model.cost = pyo.Param(model.edges, initialize={(u,v): self.g.edges[u,v]['cost'] for u, v in self.g.edges})
        model.demand = pyo.Param(model.nodes,initialize={n: G.nodes[n]['demand'] for n in G.nodes()})
        model.cap = pyo.Constraint(model.edges, rule=self.r.edge_cap_rule)
        model.flow = pyo.Constraint(self.model.nodes, rule=self.r.edge_flow_rule)


    def create_model(self, problem='max_flow'):
        if problem == 'max_flow':
            self._create_maxflow()

    def _create_maxflow(self):
        if not hasattr(self.model, 'capacity'):
            self.model.capacity = pyo.Param(
                self.model.edges,initialize={(u, v): self.g.edges[u, v]['capacity'] for u, v in self.g.edges()}
            )

        if not hasattr(self.model, 'demand'):
            self.model.demand = pyo.Param(self.model.nodes,initialize={n: self.g.nodes[n]['demand'] for n in self.g.nodes()})

        if not hasattr(self.model, 'cap'):
            self.model.cap = pyo.Constraint(self.model.edges, rule=self.r.edge_cap_rule)

        if not hasattr(self.model, 'F'):
            self.model.F = pyo.Var(within=pyo.NonNegativeReals)

        if not hasattr(self.model, 'obj'):
            self.model.obj = pyo.Objective(
                expr=self.model.F,sense=pyo.maximize
            )

    def __configure_graph(self, source='s', sink='t'):
    
        H = self.g.copy()

        if not H.is_directed():
            H = H.to_directed()

        for _,_,d in H.edges(data=True):
            d.setdefault('capacity', 1)
            d.setdefault('cost', 0)

        for n, data in H.nodes(data=True):
            data.setdefault('demand', 0)

        # Add super source/sink
        H.add_node(source, demand=0)
        H.add_node(sink, demand=0)

        supply = [n for n in H.nodes if H.nodes[n].get('demand') < 0]
        demand = [n for n in H.nodes if H.nodes[n].get('demand') > 0]
        for n in supply:
            H.add_edge(source, n, capacity=float('inf'), cost=0)
        for n in demand:
            H.add_edge(n, sink, capacity=float('inf'), cost=0)
        return H


lp = LP(G)

        
    


In [186]:
lp.g.edges()

OutEdgeView([(0, 6), (0, 7), (0, 8), (0, 9), (0, 10), (0, 't'), (1, 6), (1, 7), (1, 8), (1, 9), (1, 10), (1, 't'), (2, 6), (2, 7), (2, 8), (2, 9), (2, 't'), (3, 6), (3, 7), (3, 8), (3, 9), (4, 6), (4, 7), (4, 8), (4, 9), (4, 10), (4, 't'), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 't'), (6, 11), (6, 12), (6, 13), (7, 11), (7, 12), (7, 13), (8, 11), (8, 12), (8, 13), (9, 11), (9, 12), (9, 13), (10, 11), (10, 12), (10, 13), (11, 14), (12, 14), (13, 14), (14, 15), (15, 16), (15, 17), (16, 18), (16, 19), (16, 20), (16, 22), (17, 18), (17, 19), (17, 20), (17, 22), (18, 23), (18, 24), (18, 25), (19, 23), (19, 24), (19, 25), (20, 23), (20, 24), (20, 25), (21, 23), (21, 25), (22, 23), (22, 24), (22, 25), (23, 26), (24, 26), (25, 26), ('s', 26)])

In [187]:
if all('capacity' in d for _,_,d in G.edges(data=True)):
    print('yes')
else:
    print('no')

yes


['role', 'power', 'capacity', 'probability', 'performance', 'cost']

In [129]:
from src.lp import preprocess_demand_imbalance

G, layers = create_layered_graph(layer_sizes)
G = assign_properties(G, layer_sizes)
G = preprocess_demand_imbalance(G)

# Sets & params
model = pyo.ConcreteModel()
model.nodes = pyo.Set(initialize=list(G.nodes()))
model.edges = pyo.Set(initialize=list(G.edges()), dimen=2)
model.capacity = pyo.Param(model.edges,initialize={(u, v): G.edges[u, v]['capacity'] for u, v in G.edges()})
model.demand = pyo.Param(model.nodes,initialize={n: G.nodes[n]['demand'] for n in G.nodes()})
model.s = pyo.Set(initialize=[n for n in model.nodes if model.demand[n] < 0])
model.t = pyo.Set(initialize=[n for n in model.nodes if model.demand[n] > 0])
model.F = pyo.Var(within=pyo.NonNegativeReals)
model.obj = pyo.Objective(expr=model.F, sense=pyo.maximize)

In [175]:
print(model.obj.EXPRESSION_SYSTEM)

ExpressionType.NUMERIC


In [179]:
print(model.obj.expr)

F


In [101]:
dir(model.obj)

['ASSOCIATIVITY',
 'EXPRESSION_SYSTEM',
 'NoObjective',
 'PRECEDENCE',
 'Skip',
 '_ComponentDataClass',
 '_DEFAULT_INDEX_CHECKING_ENABLED',
 '_PPRINT_INDENT',
 '__abs__',
 '__add__',
 '__array_ufunc__',
 '__auto_slots__',
 '__autoslot_mappers__',
 '__bool__',
 '__call__',
 '__class__',
 '__contains__',
 '__deepcopy__',
 '__deepcopy_field__',
 '__deepcopy_state__',
 '__delattr__',
 '__delitem__',
 '__dict__',
 '__dir__',
 '__div__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__float__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__idiv__',
 '__imul__',
 '__init__',
 '__init_subclass__',
 '__int__',
 '__ipow__',
 '__isub__',
 '__iter__',
 '__itruediv__',
 '__le__',
 '__len__',
 '__lt__',
 '__module__',
 '__mul__',
 '__ne__',
 '__neg__',
 '__new__',
 '__pos__',
 '__pow__',
 '__radd__',
 '__rdiv__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rmul__',
 '__rpow__',
 '__rsub__',
 '__rtruediv__',
 '__setattr__